In [4]:
import logging
import multiprocessing
import os
import pickle
from functools import partial

logging.getLogger("pint").setLevel(logging.ERROR)

if os.environ.get("SLURM_CPUS_PER_TASK"):
    cores = int(os.environ.get("SLURM_CPUS_PER_TASK", 1))
else:
    cores = multiprocessing.cpu_count()
print(f"Number of cores: {cores}")

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(cores)

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.table import Table
import gpjax as gpx
from gpjax.kernels import RBF, Linear, Matern12, Matern32, Matern52, Periodic, White
from jax import jit
from jaxoplanet import orbits
from jaxoplanet.light_curves import LimbDarkLightCurve
from jaxopt import ScipyMinimize
from tensorflow_probability.substrates.jax.distributions import Normal
from tqdm import tqdm

from gallifrey.inference import (
    calculate_whitened_residuals,
    log_likelihood_function,
    predictive_distribution,
)
from gallifrey.kernelsearch import KernelSearch, get_trainables, kernel_summary
from gallifrey.mcmc import nuts_warmup, run_mcmc
from gallifrey.visualisation import (
    plot_masks,
    plot_prediction,
    plot_residuals,
    plot_allan_deviation,
)

import lightkurve as lk
import jax.numpy as jnp

Number of cores: 8


In [5]:
jax.config.update("jax_enable_x64", True)
rng_key = jax.random.PRNGKey(42)

plt.style.use("../figures/gpjax.mplstyle")

mode = "load"

## DOWNLOAD DATA

In [6]:
## This part is adapted from the exoplanet code case studies (https://gallery.exoplanet.codes/tutorials/lc-gp-transit/)

# get lightcurve, and perform some data cleaning.
lcf = lk.search_lightcurve(
    "6184894", mission="Kepler", author="Kepler", cadence="long"
).download_all()
lc = lcf.stitch().remove_nans().remove_outliers()
lc = lc[lc.quality == 0]

# normalise data
t = jnp.asarray(lc.time.value, dtype=jnp.float64)
y = jnp.asarray(lc.flux, dtype=jnp.float64)
y /= jnp.nanmedian(y)
yerr = jnp.asarray(lc.flux_err, dtype=jnp.float64)
yerr = yerr / jnp.nanmedian(y)

##kernel search mask
sample_region = jnp.where((t >= 200) & (t <= 210))
t_sample = t[sample_region]
y_sample = y[sample_region]
yerr_sample = yerr[sample_region]
dataset = gpx.Dataset(X=t_sample.reshape(-1, 1), y=y_sample.reshape(-1, 1))

## CREATE SPARSE GP

In [7]:
n_inducing = 200
inducing_points = jnp.linspace(
    t_sample.min(),
    t_sample.max(),
    n_inducing,
).reshape(-1, 1)

In [8]:
model = gpx.likelihoods.Gaussian(num_datapoints=len(t_sample)) * gpx.gps.Prior(
    mean_function=gpx.mean_functions.Zero(), kernel=RBF()
)

In [10]:
q = gpx.variational_families.CollapsedVariationalGaussian(
    posterior=model, inducing_inputs=inducing_points
)

mll = jit(gpx.objectives.ConjugateMLL(negative=True))
elbo = jit(gpx.objectives.CollapsedELBO(negative=True))

In [ ]:
gpx.fit_scipy(
    model=q,
    objective=elbo,  # type: ignore
    train_data=dataset,
    max_iters=1000,
    verbose=True,
)

In [ ]:
import optax as ox

sparse_model, history = gpx.fit(
    model=q,
    objective=elbo,
    train_data=dataset,
    optim=ox.adamw(learning_rate=3e-4),
    num_iters=10000,
    key=rng_key,
)

In [16]:
def get(m):
    x, a = gpx.fit_scipy(
        model=m,
        objective=mll,  # type: ignore
        train_data=dataset,
        max_iters=1000,
        verbose=False,
    )
    return x, a

In [21]:
get(model)

(ConjugatePosterior(prior=Prior(kernel=RBF(compute_engine=DenseKernelComputation(), active_dims=None, name='RBF', lengthscale=Array(0.5471292, dtype=float32), variance=Array(0.3680664, dtype=float32)), mean_function=Zero(constant=Array([0.], dtype=float32)), jitter=1e-06), likelihood=Gaussian(num_datapoints=426, integrator=AnalyticalGaussianIntegrator(), obs_stddev=Array(2.1361266e-13, dtype=float32)), jitter=1e-06),
 Array([  410.81031971, -2305.42195521], dtype=float64))

In [22]:
from joblib import Parallel, delayed

In [23]:
k = Parallel(n_jobs=8)(delayed(get)(i) for i in [model] * 8)

In [24]:
k[0]

(ConjugatePosterior(prior=Prior(kernel=RBF(compute_engine=DenseKernelComputation(), active_dims=None, name='RBF', lengthscale=Array(1., dtype=float32), variance=Array(1., dtype=float32)), mean_function=Zero(constant=Array([0.], dtype=float32)), jitter=1e-06), likelihood=Gaussian(num_datapoints=426, integrator=AnalyticalGaussianIntegrator(), obs_stddev=Array(1., dtype=float32)), jitter=1e-06),
 Array([410.81027, 410.81027], dtype=float32))

In [26]:
k[0][1]

Array([410.81027, 410.81027], dtype=float32)

In [3]:
import jax.numpy as jnp

In [5]:
float(-jnp.inf)

-inf